# Case study: do tree ensembles price Berlin Airbnb listings better?

*Session 10, block 3 practice. Author: course team, licence CC-BY-4.0.*

**Question.** What drives the nightly price of a Berlin Airbnb listing, and is a tree ensemble worth replacing the linear price model of Session 6? We compare models on identical host-grouped folds (Session 7), tune the challenger, decide with paired differences and a bootstrap interval, and interpret the result with permutation importance and SHAP ([theory page 4](../theory/04-interpretation-and-model-comparison.md)).

**Data.** Inside Airbnb, Berlin, snapshot of 26 June 2026 (CC BY 4.0), prepared with `uv run python case-study/prepare_airbnb.py`; no host names; report in aggregate. Leaky columns (`estimated_revenue_l365d`, `estimated_occupancy_l365d`, Session 9) stay out of the models until the leak check in Part 5.

**Setup.** LightGBM needs the OpenMP runtime on macOS (`brew install libomp`). The notebook runs in about three minutes.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.width", 120)
# find the repository root, wherever the notebook is started from
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "case-study" / "data").exists())
AIRBNB = ROOT / "case-study" / "data" / "airbnb"
import json
import re
import time

import matplotlib.pyplot as plt
import shap
from lightgbm import LGBMRegressor, early_stopping
from scipy.stats import loguniform, randint
from sklearn.ensemble import HistGradientBoostingRegressor, RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.model_selection import GroupKFold, GroupShuffleSplit, RandomizedSearchCV
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

## The data and two feature sets

- **Session 6 features:** guests, room type, distance to Alexanderplatz, district (the current model).
- **All features:** size, bedrooms, bathrooms, location, property type, minimum stay, availability, reviews, 122 common amenities and the number of amenities.

In [ ]:
listings = pd.read_parquet(AIRBNB / "listings.parquet")
# short-stay listings with a price (minimum stay < 28 nights, as in Sessions 4-6),
# without the 26 outliers priced below EUR 10 or above EUR 1,000
bnb = listings[(listings["minimum_nights"] < 28) & listings["price"].between(10, 1000)].reset_index(drop=True)
bnb["dist_km"] = np.hypot((bnb["latitude"] - 52.5219) * 111.2,      # km to Alexanderplatz
                          (bnb["longitude"] - 13.4132) * 68.0)
print(len(listings), "listings in the snapshot,", len(bnb), "in the price-model table,",
      bnb["host_id"].nunique(), "hosts")
amen_lists = bnb["amenities"].map(json.loads)
common = amen_lists.explode().value_counts().loc[lambda c: c >= 200].index
amen = pd.DataFrame({"am_" + re.sub(r"\W+", "_", a).strip("_").lower(): amen_lists.map(lambda x, a=a: a in x).astype(int)
                     for a in common})
y, price, hosts = np.log(bnb["price"]), bnb["price"], bnb["host_id"]

X_s6 = pd.get_dummies(bnb[["accommodates", "dist_km", "room_type", "district"]], dtype=int)
rich_num = ["accommodates", "bedrooms", "beds", "bathrooms", "dist_km", "latitude", "longitude",
            "minimum_nights", "availability_365", "number_of_reviews", "review_scores_rating"]
X_rich = pd.concat([bnb[rich_num], bnb[["room_type", "district", "property_type"]].astype("category"),
                    amen.assign(n_amenities=amen_lists.map(len))], axis=1)
X_dummies = pd.get_dummies(X_rich, dtype=int)
print(X_s6.shape, X_rich.shape, X_dummies.shape)

## Part 1 · Five models on identical folds

In [ ]:
def lgbm(**params):
    base = dict(n_estimators=600, learning_rate=0.03, num_leaves=31, subsample=0.8, subsample_freq=1,
                colsample_bytree=0.5, verbose=-1, random_state=0)
    return LGBMRegressor(**{**base, **params})


models = {
    "current: linear, Session 6 features": (LinearRegression(), X_s6),
    "ridge, all features": (make_pipeline(SimpleImputer(strategy="median", add_indicator=True), StandardScaler(),
                                          Ridge(alpha=10)), X_dummies),
    "random forest, all features": (RandomForestRegressor(300, min_samples_leaf=3, max_features=0.33,
                                                          n_jobs=-1, random_state=0), X_dummies),
    "HistGradientBoosting, all features": (HistGradientBoostingRegressor(categorical_features="from_dtype",
                                                                        random_state=0), X_rich),
    "challenger: LightGBM, all features": (lgbm(), X_rich),
}
folds = list(GroupKFold(5).split(X_rich, y, groups=hosts))
fold_mae, rows = {}, []
for name, (model, X_m) in models.items():
    t0, pred, fold_mae[name] = time.time(), np.zeros(len(y)), []
    for tr, va in folds:
        pred[va] = model.fit(X_m.iloc[tr], y.iloc[tr]).predict(X_m.iloc[va])
        fold_mae[name].append(mean_absolute_error(price.iloc[va], np.exp(pred[va])))
    rows.append({"model": name, "R² (log)": r2_score(y, pred), "MAE €": mean_absolute_error(price, np.exp(pred)),
                 "median AE €": np.median(np.abs(price - np.exp(pred))), "seconds": time.time() - t0})
results = pd.DataFrame(rows).set_index("model")
results.round(3)

**Task 1.** How much of the gain over the current model comes from the features (compare ridge with the linear model) and how much from the model type (compare the ensembles with ridge)? Add a row "LightGBM, Session 6 features" and explain the result.

## Part 2 · Tuning the challenger

### Early stopping on held-out hosts

In [ ]:
dev, test = next(GroupShuffleSplit(1, test_size=0.2, random_state=0).split(X_rich, groups=hosts))
fit_idx, stop_idx = next(GroupShuffleSplit(1, test_size=0.2, random_state=1).split(dev, groups=hosts.iloc[dev]))
fit_idx, stop_idx = dev[fit_idx], dev[stop_idx]
es = lgbm(n_estimators=3000, learning_rate=0.03).fit(
    X_rich.iloc[fit_idx], y.iloc[fit_idx], eval_set=[(X_rich.iloc[stop_idx], y.iloc[stop_idx])],
    eval_metric="l1", callbacks=[early_stopping(100, verbose=False)])
print("best number of trees:", es.best_iteration_)

### Randomised search with host-grouped folds

In [ ]:
space = {"learning_rate": loguniform(0.01, 0.2), "num_leaves": randint(8, 128), "min_child_samples": randint(5, 100),
         "colsample_bytree": [0.3, 0.5, 0.8], "reg_lambda": loguniform(1e-3, 10)}
search = RandomizedSearchCV(lgbm(n_jobs=1), space, n_iter=12, cv=GroupKFold(4), scoring="neg_mean_absolute_error",
                            random_state=0, n_jobs=-1)
search.fit(X_rich.iloc[dev], y.iloc[dev], groups=hosts.iloc[dev])
res = pd.DataFrame(search.cv_results_).sort_values("rank_test_score")
cols = [c for c in res.columns if c.startswith("param_")] + ["mean_test_score", "std_test_score"]
print("default LightGBM, same folds:",
      round(-np.mean([mean_absolute_error(y.iloc[dev].iloc[va], lgbm().fit(X_rich.iloc[dev].iloc[tr], y.iloc[dev].iloc[tr])
                                         .predict(X_rich.iloc[dev].iloc[va]))
                      for tr, va in GroupKFold(4).split(X_rich.iloc[dev], groups=hosts.iloc[dev])]), 4))
res[cols].rename(columns=lambda c: c.replace("param_", "")).round(4).head(6)

The score is the negative MAE on the **log** scale: −0.25 means that a typical prediction is off by roughly a quarter of the price. **Task 2.** Is the best setting clearly better than the default, given the standard deviations? Which would you keep, and why?

## Part 3 · Current model or challenger?

Paired fold differences from Part 1, and a bootstrap interval of the MAE difference on held-out hosts, resampling **hosts** (cluster bootstrap, Session 7).

In [ ]:
d = np.array(fold_mae["challenger: LightGBM, all features"]) - np.array(fold_mae["current: linear, Session 6 features"])
print("MAE difference per fold (EUR):", d.round(1), " mean", round(d.mean(), 1), " sd", round(d.std(ddof=1), 1))

current = LinearRegression().fit(X_s6.iloc[dev], y.iloc[dev])
challenger = lgbm().fit(X_rich.iloc[dev], y.iloc[dev])
err_cur = np.abs(price.iloc[test].to_numpy() - np.exp(current.predict(X_s6.iloc[test])))
err_ch = np.abs(price.iloc[test].to_numpy() - np.exp(challenger.predict(X_rich.iloc[test])))
print("test MAE: current", round(err_cur.mean(), 1), " challenger", round(err_ch.mean(), 1))

diff_by_host = pd.DataFrame({"d": err_ch - err_cur, "host": hosts.iloc[test].to_numpy()}).groupby("host")["d"]
sums, counts = diff_by_host.sum().to_numpy(), diff_by_host.size().to_numpy()
rng = np.random.default_rng(0)
boot = []
for _ in range(2000):
    pick = rng.integers(0, len(sums), len(sums))
    boot.append(sums[pick].sum() / counts[pick].sum())
print("95% interval of the MAE difference (EUR):", np.percentile(boot, [2.5, 97.5]).round(1))

**Task 3.** Write the decision table of theory page 4, Section 3: MAE with interval, fit time (Part 1), prediction time, dependencies, explainability. Would you replace the current model? For which users does a typical error of the size of the median absolute error still matter?

## Part 4 · What does the challenger rely on?

In [ ]:
perm = permutation_importance(challenger, X_rich.iloc[test], y.iloc[test], scoring="r2", n_repeats=5, random_state=0)
imp = pd.Series(perm.importances_mean, index=X_rich.columns)
print("all amenities together:", round(imp.filter(like="am_").sum(), 3))
imp.sort_values(ascending=False).head(10).round(3)

Shuffling the 122 amenity columns one at a time underestimates them as a group (they are correlated). **Task 4a.** Shuffle all amenity columns *together* (the same permutation of rows for the whole block) and compare the drop in R² with the sum above.

In [ ]:
sv = shap.TreeExplainer(challenger)(X_rich.iloc[test])
shap.plots.beeswarm(sv, max_display=12, show=False)
plt.title("SHAP values (log price) on held-out hosts")
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, col in zip(axes, ["dist_km", "accommodates"]):
    j = list(X_rich.columns).index(col)
    ax.scatter(X_rich.iloc[test][col], 100 * (np.exp(sv.values[:, j]) - 1), s=6, alpha=0.4)
    ax.axhline(0, color="grey", lw=1)
    ax.set_xlabel(col)
    ax.set_ylabel("SHAP contribution as price change (%)")
plt.show()

In [ ]:
i = 0
print(X_rich.iloc[test[i]][["room_type", "property_type", "accommodates", "bedrooms", "dist_km"]].to_dict())
print("base price", round(float(np.exp(sv.base_values[i]))), "-> prediction",
      round(float(np.exp(sv.base_values[i] + sv.values[i].sum()))), "| actual", round(float(price.iloc[test[i]])))
shap.plots.waterfall(sv[i], max_display=8)

**Task 4b.** Explain the waterfall of one listing in euros for a host, using the factors `exp(φ)`. Which statement about this listing would be wrong to make ("if you added a dishwasher, your price would rise by …")?

## Part 5 · A leak check with SHAP

Add the leaky revenue and occupancy columns of Session 9 and look at the same tools.

In [ ]:
X_leak = X_rich.assign(revenue=bnb["estimated_revenue_l365d"], occupancy=bnb["estimated_occupancy_l365d"])
leaky = lgbm().fit(X_leak.iloc[dev], y.iloc[dev])
print("held-out R² without / with the leak:",
      round(r2_score(y.iloc[test], challenger.predict(X_rich.iloc[test])), 3),
      round(r2_score(y.iloc[test], leaky.predict(X_leak.iloc[test])), 3))
sv_leak = shap.TreeExplainer(leaky)(X_leak.iloc[test])
pd.Series(np.abs(sv_leak.values).mean(axis=0), index=X_leak.columns).nlargest(6).round(3)

**Task 5.** Which signals in the output would make you suspicious even if you did not know how Inside Airbnb computes revenue?

**Team project.** For the interim review, prepare one slide with your current model, the strongest challenger, and the comparison table (metric with interval, cost, explainability).